#### Primeiro, vamos analizar o enunciado do problema:
1- Um conjunto cilindro-pistão contém inicialmente 5 mol de n-pentano a 500 K e 3 bar. O gás
é comprimido isotermicamente até 30 bar, com um trabalho fornecido ao sistema de 10 kJ.
Durante esse processo, ocorre transferência de calor com a vizinhança, que se encontra a uma
temperatura de 298 K.
Determine se esse processo é reversível, irreversível ou impossível. Admita que o n-pentano
obedeça à equação de estado de Peng-Robinson.

Temos os seguintes valores: 
- T1 = 500 K = T2 (processo isotérmico)
- P1 = 3 bar
- P2 = 30 bar
- W = 10 kJ
- T<sub>externa</sub> = 298K


Queremos saber se o processo é reverssível, impossível ou irreverssível. Para tanto, precisamos saber o valor de S<sub>gen</sub>

Antes de começarmos, vamos importar todas os módulos que serão necessários para a resolução da questão

In [1]:
from substancias import npentano
from eos import PR
from propriedades import entropia, energia_interna
from balancos import balanco_energia, balanco_entropia
from constantes import BAR

In [2]:
T1 = 500
T2 = T1
P1 = 3 * BAR
P2 = 30 * BAR
W = 10*10**3
n = 5
Text = 298

Vamos calcular os valores de Z e descobrir se o processo permanece inteiramente gasoso durante a compressão

In [3]:
eos_pentano = PR(npentano) # Instanciando a equação de estado de Peng-Robinson para o n-pentano.
Z1 = eos_pentano.calcZ(T1,P1) # Calculando o fator de compressibilidade para o n-pentano a 400 K e 2 MPa.
Z2 = eos_pentano.calcZ(T2,P2) # Calculando o fator de compressibilidade para o n-pentano a 400 K e 2 MPa.
print(f"Z1 = {Z1}")
print(f"Z2 = {Z2}")

Z1 = [np.float64(0.971889658638591)]
Z2 = [np.float64(0.6786252871355279)]


Como podemos ver, há apenas um valor para Z1 e Z2. Logo, pode-se intuir que o processo tem a presença de gases do começo ao final

O balanço de entropia pode ser descrito pela equação: 
- $\Delta$ S<sub>total</sub> = m $\Delta$ S<sub>1-2</sub> + Q/T + S<sub>gen</sub>

Hipóteses: 

1- Sistema fechado (m $\Delta$ S<sub>1-2</sub> = 0)

Logo, a sua equação será na forma:

- $\Delta$ S<sub>total</sub> = Q/T + S<sub>gen</sub>


Já o balanço de energia pode ser descrito pela equação:
- $\Delta$ U = m $\Delta$ H<sub>1-2</sub> + Q + W

Hipóteses: 

1- Sistema fechado (m $\Delta$ H<sub>1-2</sub> = 0)

Logo, podemos chegar na expressão: 
- $\Delta$ U = Q + W

Como temos o valor do trabalho, podemos calcular o valor energia interna a partir das propriedades residuais, e então descobrir o calor, que posteriormente será importante para o cálculo da entropia.

De acordo com as propriedades residuais, a energia interna pode ser dada pela equação: 
- U<sub>2</sub> - U<sub>1</sub> = - U<sup>R</sup><sub>1</sub> + U<sup>id</sup><sub>2</sub> - U<sup>id</sup><sub>1</sub> + U<sup>R</sup><sub>2</sub>

Para calcularmos o valor, podemos usar o módulo das equações residuais construído pelo autor. Dessa forma:

In [5]:
Ures1 = energia_interna().residual(eos_pentano, T1, P1, n, 'vapor') # Calculando a energia interna residual para o n-pentano a 400 K e 2 MPa.
Ures2 = energia_interna().residual(eos_pentano, T2, P2, n, 'vapor') # Calculando a energia interna residual para o n-pentano a 400 K e 2 MPa.
Uideal = energia_interna().ideal(npentano, T1, T2, n) # Calculando a energia interna ideal para o n-pentano a 400 K e 2 MPa.
Utotal = -Ures1 + Uideal + Ures2
print(f"-Ures1 = {-Ures1} + Uideal = {Uideal} + Ures2 = {Ures2} = Utotal = {Utotal}")


-Ures1 = 1303.846550095342 + Uideal = 0.0 + Ures2 = -17237.601723276974 = Utotal = -15933.755173181631


Agora podemos voltar na expressão do balanço de energia e achar o valor do trabalho
- $\Delta$ U - W = Q

In [7]:
Q = Utotal - W # Calculando o calor trocado com a vizinhança.
print(f"Q = {Q}")

Q = -25933.75517318163


Agora podemos calcular a entropia residual. De forma análoga a energia interna:
- S<sub>2</sub> - S<sub>1</sub> = - S<sup>R</sup><sub>1</sub> + S<sup>id</sup><sub>2</sub> - S<sup>id</sup><sub>1</sub> + S<sup>R</sup><sub>2</sub>

In [9]:
Sres1 = entropia().residual(eos_pentano, T1, P1, n, 'vapor') # Calculando a entropia residual para o n-pentano a 400 K e 2 MPa.
Sres2 = entropia().residual(eos_pentano, T2, P2, n, 'vapor') # Calculando a entropia residual para o n-pentano a 400 K e 2 MPa.
Sideal = entropia().ideal(npentano, T1, T2, P1, P2, n) # Calculando a entropia ideal para o n-pentano a 400 K e 2 MPa.
Stotal = -Sres1 + Sideal + Sres2
print(f"-Sres1 = {-Sres1} + Sideal = {Sideal} + Sres2 = {Sres2} = Stotal = {Stotal}")

-Sres1 = 2.613840502910519 + Sideal = -95.72306748594846 + Sres2 = -35.50276547443727 = Stotal = -128.6119924574752


Podemos agora, voltar a equação:
- $\Delta$ S<sub>total</sub> = Q/T + S<sub>gen</sub>

In [10]:
Sgen = Stotal - Q/Text # Calculando a entropia gerada no processo.
print(f"Sgen = {Sgen}")

Sgen = -41.58596838639589


Logo, o processo é impossível